# Snap2 Cursive OCR Fine-Tuning (PARSeq -> ONNX)

**Zero LLM tokens required.** Run this notebook on a free **Google Colab T4 GPU** (or Kaggle GPU) to fine-tune `baudm/parseq` on your Snap2 cursive dataset (`training/dataset/`) and export a drop-in `model.onnx` for `Snap2/public/models/parseq/model.onnx`.

In [ ]:
# 1. Verify GPU and install dependencies
!nvidia-smi
!pip install -q timm==0.9.16 pytorch-lightning==2.2.4 onnx onnxruntime lmdb pillow

In [ ]:
# 2. Clone official PARSeq repo
!git clone --depth 1 https://github.com/baudm/parseq.git
%cd parseq

In [ ]:
# 3. Upload or mount your zipped Snap2 training/dataset folder
# Expected inside dataset.zip: splits/train.tsv, splits/val.tsv, real/, synth/
from google.colab import files
import zipfile, pathlib

uploaded = files.upload()  # Upload dataset.zip
for fname in uploaded:
    with zipfile.ZipFile(fname, 'r') as zf:
        zf.extractall('/content/snap2_dataset')
print('Extracted to /content/snap2_dataset')

In [ ]:
# 4. Fine-tune pretrained PARSeq on Snap2 cursive manifests (no horizontal flips!)
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = torch.hub.load('baudm/parseq', 'parseq', pretrained=True).to(device)
model.train()

img_transform = T.Compose([
    T.Resize((32, 128), T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(0.5, 0.5),
])

class TsvOcrDataset(Dataset):
    def __init__(self, tsv_path):
        p = Path(tsv_path)
        self.base = p.parent
        self.items = []
        for i, line in enumerate(p.read_text(encoding='utf-8').splitlines()):
            if i == 0 and line.startswith('rel_path\t'):
                continue
            parts = line.split('\t')
            if len(parts) >= 2:
                ip = (self.base / parts[0]).resolve()
                if ip.is_file():
                    self.items.append((ip, parts[1].strip()))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        ip, label = self.items[idx]
        img = Image.open(ip).convert('RGB')
        return img_transform(img), label

train_ds = TsvOcrDataset('/content/snap2_dataset/splits/train.tsv')
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
epochs = 10
for epoch in range(1, epochs + 1):
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        optimizer.zero_grad()
        loss = model.training_step((imgs, list(labels)), 0)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 20.0)
        optimizer.step()
        total_loss += float(loss.item())
    print(f'Epoch {epoch}/{epochs} | Loss: {total_loss / max(1, len(train_loader)):.4f}')

In [ ]:
# 5. Export drop-in ONNX model for Snap2 (public/models/parseq/model.onnx)
model.eval().cpu()
dummy = torch.randn(1, 3, 32, 128)
out_onnx = '/content/parseq_cursive.onnx'
torch.onnx.export(
    model,
    dummy,
    out_onnx,
    input_names=['images'],
    output_names=['logits'],
    dynamic_axes={'images': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)
print('Exported:', out_onnx)
files.download(out_onnx)